In [16]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix
)

import time
import os
from pathlib import Path

Check GPU

In [17]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.0+cu130
CUDA available: True
CUDA: 13.0
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


GPU Optimization

In [18]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    torch.set_float32_matmul_precision("high")

print("Using:", device)

Using: cuda


Importing the Data

In [19]:
DATA_DIR = Path(r"C:\Users\badri\OneDrive\Desktop\GIS-intra-iit\GeoShift\data")

train_images = np.load(DATA_DIR / "train_images.npy")
test_images = np.load(DATA_DIR / "test_images.npy")

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

In [20]:
train_float = train_images.astype(np.float32) / 255.0

channel_mean = train_float.mean(
    axis=(0, 2, 3)
)

channel_std = train_float.std(
    axis=(0, 2, 3)
)

del train_float

print("Mean:", channel_mean)
print("Std :", channel_std)

Mean: [0.1651127  0.18718533 0.19730262 0.32161716]
Std : [0.04854234 0.05274455 0.06854094 0.10061419]


In [21]:
np.save("channel_mean.npy", channel_mean)
np.save("channel_std.npy", channel_std)

In [22]:
indices = np.arange(len(train_df))

labels = train_df["label"].values

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

print("Training samples  :", len(train_idx))
print("Validation samples:", len(val_idx))

Training samples  : 27151
Validation samples: 6788


Importing the Dataset Model

In [23]:
import importlib.util
import sys

module_name = "GeoShiftDataset"
file_path = r"C:\Users\badri\OneDrive\Desktop\GIS-intra-iit\GeoShift\src\Dataset.py"
spec = importlib.util.spec_from_file_location(module_name, file_path)
my_module = importlib.util.module_from_spec(spec)
sys.modules[module_name] = my_module
spec.loader.exec_module(my_module)

GeoShiftDataset = getattr(my_module, "GeoShiftDataset")

In [24]:
train_dataset = GeoShiftDataset(
    train_images[train_idx],
    train_df["label"].values[train_idx],
    mean=channel_mean,
    std=channel_std,
    augment=True
)

val_dataset = GeoShiftDataset(
    train_images[val_idx],
    train_df["label"].values[val_idx],
    mean=channel_mean,
    std=channel_std,
    augment=False
)

DataLoaders

In [25]:
BATCH_SIZE = 512

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

### Defining the Model

In [26]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(4, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Linear(128, 6)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

Load the Checkpoint in the file saved here

In [27]:
checkpoint = torch.load(
    "baseline_random_split_3_49PM_23_09_2026.pth",
    map_location=device,
    weights_only=False
)

model = SimpleCNN().to(device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

channel_mean = checkpoint["channel_mean"]
channel_std = checkpoint["channel_std"]

print("Loaded model")
print("Previous Macro F1:", checkpoint["macro_f1"])

Loaded model
Previous Macro F1: 0.9915


### Testing

In [57]:
from sklearn.model_selection import train_test_split

indices = np.arange(len(train_df))
labels = train_df["label"].values

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=36,
    stratify=labels
)

print("Training:", len(train_idx))
print("Test:", len(test_idx))

Training: 27151
Test: 6788


In [58]:
test_dataset = GeoShiftDataset(
    train_images[test_idx],
    labels[test_idx],
    mean=channel_mean,
    std=channel_std,
    augment=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

Prediction Score

In [59]:
model.eval()

all_predictions = []
all_targets = []
all_probabilities = []

with torch.no_grad():

    for images, labels_batch in test_loader:

        images = images.to(device, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(images)

        probabilities = torch.softmax(outputs, dim=1)
        predictions = probabilities.argmax(dim=1)

        all_predictions.append(
            predictions.cpu().numpy()
        )

        all_targets.append(
            labels_batch.numpy()
        )

        all_probabilities.append(
            probabilities.cpu().numpy()
        )

predictions = np.concatenate(all_predictions)
targets = np.concatenate(all_targets)
probabilities = np.concatenate(all_probabilities)

print("Predictions:", predictions.shape)
print("Targets:", targets.shape)

Predictions: (6788,)
Targets: (6788,)


In [60]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

accuracy = accuracy_score(targets, predictions)

macro_f1 = f1_score(
    targets,
    predictions,
    average="macro"
)

print("Accuracy :", accuracy)
print("Macro F1 :", macro_f1)

Accuracy : 0.9949911608721272
Macro F1 : 0.9931120357701895


In [61]:
class_names = [
    "Forest",
    "Shrubland",
    "Grassland",
    "Cropland",
    "Built-up",
    "Water/Wetland"
]

print(
    classification_report(
        targets,
        predictions,
        target_names=class_names,
        digits=4
    )
)

               precision    recall  f1-score   support

       Forest     0.9946    0.9987    0.9966      1483
    Shrubland     0.9988    0.9881    0.9934       837
    Grassland     0.9842    0.9904    0.9873       941
     Cropland     0.9861    0.9775    0.9818       579
     Built-up     0.9991    1.0000    0.9996      2318
Water/Wetland     1.0000    1.0000    1.0000       630

     accuracy                         0.9950      6788
    macro avg     0.9938    0.9924    0.9931      6788
 weighted avg     0.9950    0.9950    0.9950      6788



We need to check if it is overfitting or not, so we're doing a test where we are training for set of regions, and asking whether other regions will go for it

In [62]:
train_df["region"] = train_df["Id"].str.extract(r"GEO_(R\d+)_")

In [64]:
from sklearn.metrics import f1_score, accuracy_score

region_results = []

for region in sorted(train_df["region"].unique()):

    region_mask = train_df["region"].values[test_idx] == region

    if region_mask.sum() == 0:
        continue

    y_true = targets[region_mask]
    y_pred = predictions[region_mask]

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    region_results.append({
        "Region": region,
        "Samples": len(y_true),
        "Accuracy": accuracy,
        "Macro_F1": macro_f1
    })

region_results = pd.DataFrame(region_results)

print(region_results)

TypeError: '<' not supported between instances of 'float' and 'str'